### Convert h5ad to parquet and split

In [1]:
import os
from deeptan.utils.data import (
    read_h5ad,
    h5ad_to_parquet,
    JointStratifiedSplitter,
)

/home/wuch/miniforge3/envs/deeptan_test/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### If you want to perform 5 random splits on your data

In [2]:
seeds = [i + 42 for i in range(5)]  # Replace 5 with the actual number of random splits.

In [3]:
path_h5ad = "../data/snRNA/ath_pretrain.h5ad" # /Path/to/your_raw_data.h5ad

#### h5ad to parquet

In [4]:
path_parquet = path_h5ad.replace(".h5ad", ".parquet")
if not os.path.exists(path_parquet):
    h5ad_to_parquet(path_h5ad, path_parquet, True)

2026-05-09 22:14:36.493 | INFO     | deeptan.utils.data:h5ad_to_parquet:609 - Read ../data/snRNA/ath_pretrain.h5ad with 89654 cells and 18380 features.
2026-05-09 22:14:36.493 | INFO     | deeptan.utils.data:h5ad_to_parquet:610 - Saving to ../data/snRNA/ath_pretrain.parquet...
2026-05-09 22:14:37.330 | INFO     | deeptan.utils.data:adata_to_parquet:552 - X shape: (89654, 18380)
2026-05-09 22:14:43.580 | INFO     | deeptan.utils.data:adata_to_parquet:564 - DataFrame shape: (89654, 18381)
2026-05-09 22:14:43.597 | INFO     | deeptan.utils.data:adata_to_parquet:565 - Head of DataFrame:
shape: (5, 18_381)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01010 ┆ AT1G01020 ┆ AT1G01030 ┆ … ┆ AT5G02895 ┆ AT5G42965 ┆ AT5G44585 ┆ AT5G6362 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ 5        │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       

In [5]:
celltypes = read_h5ad(path_h5ad).obs["CellType"].to_list()
batch_id = read_h5ad(path_h5ad).obs["batch"].to_list()

print(f"Number of cell types: {len(set(celltypes))}")
print(f"{set(celltypes)}")
print(f"Number of batches: {len(set(batch_id))}")
print(f"{set(batch_id)}")

Number of cell types: 29
{'seed_Seed_coat', 'stem_Mesophyll', 'flower_Pollen', 'rosette_Meristematic', 'silique_Meristematic', 'silique_Epidermal', 'stem_Guard', 'flower_Epidermal', 'silique_Seed_(silique)', 'seedling_Stele', 'stem_Epidermal', 'seedling_Mesophyll', 'seed_Guard', 'seedling_Epidermal', 'rosette_Epidermal', 'stem_Stele', 'seed_Stele', 'silique_Mature_silique', 'rosette_Mesophyll', 'silique_Stele', 'seed_Meristematic', 'seed_Epidermal', 'flower_Stele', 'rosette_Guard', 'silique_Young_silique', 'rosette_Stele', 'silique_Guard', 'flower_Guard', 'seedling_Guard'}
Number of batches: 50
{'rosette-4', 'seed-1', 'seedling-2', 'seedling-18', 'stem-2', 'stem-1', 'seedling-14', 'rosette-5', 'flower-1', 'silique-1', 'seedling-3', 'silique-8', 'seed-3', 'seedling-22', 'rosette-3', 'rosette-6', 'silique-2', 'silique-4', 'seedling-6', 'rosette-1', 'seedling-8', 'stem-3', 'seedling-17', 'seedling-4', 'seedling-16', 'flower-6', 'flower-4', 'seedling-12', 'seedling-15', 'seedling-7', 'sili

#### Generate train/validation/test sets

In [6]:
def _tmp_run(_ratio, _seeds, _suff: str):
    output_dir = path_parquet.replace(".parquet", "") + ".split" + "_" + _suff

    _splitter = JointStratifiedSplitter(
        cell_types=celltypes,
        orig_idents=batch_id,
        parquet_file=path_parquet,
        output_dir=output_dir,
        ratio=_ratio,
        seeds=_seeds,
        balance_strategy="none",
    )

    _splitter.execute()      

In [7]:
_tmp_run([0.8] + [0.1, 0.1], seeds, "full")  # 80% train, 10% validation, 10% test

2026-05-09 22:15:24.692 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 0 (seed 42) with 71725 samples
2026-05-09 22:15:25.503 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 1 (seed 42) with 8961 samples
2026-05-09 22:15:26.259 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 2 (seed 42) with 8968 samples
2026-05-09 22:15:29.712 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 0 (seed 43) with 71725 samples
2026-05-09 22:15:30.400 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 1 (seed 43) with 8961 samples
2026-05-09 22:15:31.145 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 2 (seed 43) with 8968 samples
2026-05-09 22:15:34.403 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 0 (seed 44) with 71725 samples
2026-05-09 22:15:35.089 | SUCCESS  | deeptan.utils.data:_save_splits:886 - Created split 1 (seed 44) with 8961 samples
2026-05-09 22:15:35.772 | SUCCESS  | deeptan.